# Optical conductivity and dielectric function from laser and current traces

Takes a probe vector potential $A(t)$ and the induced total current $J(t)$ from a real-time simulation, and computes

$$E(t) = -\frac{1}{c}\frac{\partial A}{\partial t}, \qquad \sigma(\omega) = \frac{J(\omega)}{E(\omega)}, \qquad \epsilon(\omega) = 1 + \frac{4\pi i\,\sigma(\omega)}{\omega}$$

with the damped Fourier transform $F(\omega) = \int F(t)\, e^{i\omega t - \kappa t}\, dt$ from the `twoband` package.

**Required data files** (in `data/`):

| File | Columns used |
|---|---|
| `prim_15153_laser` | 1 = time (a.u.), 2 = A(t) |
| `prim_15153_total_current` | 1 = time (a.u.), 2 = J(t) |

In [ ]:
from pathlib import Path

import numpy as np
from matplotlib import pyplot as plt

import twoband as tb

## Settings

In [ ]:
DATA_DIR = Path("data")

CARRIER = 163.10553149492998 / tb.HARTREE_TO_EV  # probe carrier frequency (K edge), Hartree
KAPPA = 0.25 / tb.HARTREE_TO_EV  # damping in the Fourier transform

# Energy window: +/- 50 % around the carrier
ENERGY = np.linspace(0.5 * CARRIER, 1.5 * CARRIER, 5000)
ENERGY_EV = ENERGY * tb.HARTREE_TO_EV


def load_trace(name):
    # Load one simulation output file from DATA_DIR with a clear error if it is missing.
    path = DATA_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"{path} not found. See the Data section of the README.")
    return np.loadtxt(path)

## Load laser and current

In [ ]:
laser = load_trace("prim_15153_laser")
current_data = load_trace("prim_15153_total_current")

time = current_data[:, 1]
current = current_data[:, 2]

# The laser file can run longer than the current; cut it to the same time steps
n_t = len(time)
laser_time = laser[:n_t, 1]
vector_potential = laser[:n_t, 2]
if not np.allclose(laser_time, time):
    raise ValueError("laser and current files are not on the same time grid")

dt = time[1] - time[0]
field = -np.gradient(vector_potential, dt) / tb.C_AU

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(6, 7), sharex=False)
axs[0].plot(time, vector_potential, label="A(t)")
axs[0].set_xlim(0, 50)
axs[0].set_ylabel("A(t)")
axs[0].set_title(r"$\sin^4$ probe pulse")
axs[1].plot(time, field, color="r", label="E(t)")
axs[1].set_xlim(0, 50)
axs[1].set_ylabel("E(t)")
axs[2].plot(time, current, label="J(t)")
axs[2].set_ylabel("J(t)")
axs[2].set_xlabel("Time (a.u.)")
plt.tight_layout()
plt.show()

## Optical conductivity

$$\sigma(\omega) = \frac{J(\omega)}{E(\omega)}$$

In [ ]:
current_w = tb.damped_ft(time, current, ENERGY, KAPPA)
field_w = tb.damped_ft(time, field, ENERGY, KAPPA)
sigma = current_w / field_w

plt.plot(ENERGY_EV, -np.real(sigma), label="Re")
plt.plot(ENERGY_EV, -np.imag(sigma), label="Im")
plt.xlabel("Energy (eV)")
plt.ylabel(r"$\sigma(\omega)$")
plt.legend()
plt.show()

fig, axs = plt.subplots(2, 1, figsize=(6, 5), sharex=True)
axs[0].plot(ENERGY_EV, np.abs(current_w), label="|J(w)|")
axs[0].set_ylabel("FT (arb. units)")
axs[0].legend()
axs[1].plot(ENERGY_EV, np.log(np.abs(current_w) ** 2), "r", label="log |J(w)|^2")
axs[1].set_ylabel("log FT")
axs[1].set_xlabel("Energy (eV)")
axs[1].legend()
plt.show()

## Dielectric function

$$\epsilon(\omega) = 1 + \frac{4\pi i\,\sigma(\omega)}{\omega}$$

The sign follows the convention used for $\sigma$ above.

In [ ]:
im_eps = -np.imag(1 + 4j * np.pi * sigma / ENERGY)

plt.plot(ENERGY_EV, im_eps, color="r", label=r"Im[$\epsilon(\omega)$]")
plt.xlabel("Energy (eV)")
plt.ylabel(r"Im[$\epsilon(\omega)$]")
plt.legend()
plt.show()

plt.plot(ENERGY_EV, -np.real(sigma), label=r"Re[$\sigma(\omega)$]")
plt.plot(ENERGY_EV, im_eps, color="r", label=r"Im[$\epsilon(\omega)$]")
plt.xlabel("Energy (eV)")
plt.legend()
plt.show()